# Evaluation: Base vs Fine-Tuned Qwen 2.5 3B
### Invoice Extraction — Field-Level Accuracy Comparison

**Instructions:**
1. Upload `eval_locked.jsonl` (from your `data/` folder) to the Colab file browser.
2. Upload your entire `lora_model/` folder contents (or re-use the one already in Colab if session is still alive).
3. Go to **Runtime → Change runtime type → T4 GPU**.
4. Run all cells.


In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps triton

In [ ]:
import json, re, torch
from pathlib import Path
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# ── Config ──────────────────────────────────────────────────────────────
BASE_MODEL      = "unsloth/Qwen2.5-3B-Instruct"
LORA_PATH       = "lora_model"          # folder you uploaded
EVAL_FILE       = "eval_locked.jsonl"   # file you uploaded
MAX_SEQ_LENGTH  = 4096
MAX_NEW_TOKENS  = 1024
# ────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = (
    "You are a structured data extraction assistant. Your task is to extract invoice\n"
    "and receipt information from plain-text documents and return the result as a\n"
    "valid JSON object — nothing else.\n\n"
    "Rules:\n"
    "- Return ONLY the JSON object. No explanation, no markdown fences, no extra text.\n"
    "- Use null for any field not present in the document.\n"
    "- Normalize dates to YYYY-MM-DD format regardless of the source format.\n"
    "- tax_rate must be a decimal fraction (0.08 for 8%, not 8.0).\n"
    "- currency must be a 3-letter ISO 4217 code (USD, EUR, GBP, CAD, etc.).\n"
    "- line_items must contain at least one entry."
)

# Load eval set
eval_examples = [json.loads(l) for l in Path(EVAL_FILE).read_text(encoding='utf-8').splitlines() if l.strip()]
print(f"Loaded {len(eval_examples)} eval examples.")

In [ ]:
# ── Scoring helpers ──────────────────────────────────────────────────────

SCALAR_FIELDS = [
    "vendor_name", "vendor_address", "invoice_number", "invoice_date",
    "due_date", "bill_to", "subtotal", "tax_amount", "tax_rate",
    "total_amount", "currency", "payment_terms", "notes"
]

def normalize(v):
    """Lowercase + strip for string comparison."""
    if v is None:
        return None
    if isinstance(v, str):
        return v.strip().lower()
    if isinstance(v, float):
        return round(v, 4)
    return v

def score_line_items(pred_items, gt_items):
    """Compute a simple item-level recall: fraction of GT items matched."""
    if not gt_items:
        return 1.0
    if not pred_items:
        return 0.0
    matched = 0
    for gt in gt_items:
        for pred in pred_items:
            # Match on description + total
            if (normalize(pred.get('description')) == normalize(gt.get('description')) and
                    normalize(pred.get('total')) == normalize(gt.get('total'))):
                matched += 1
                break
    return matched / len(gt_items)

def score_example(pred_json, gt_json):
    """Returns dict of per-field scores (1.0 = correct, 0.0 = wrong)."""
    scores = {}
    for field in SCALAR_FIELDS:
        pred_val = normalize(pred_json.get(field))
        gt_val   = normalize(gt_json.get(field))
        scores[field] = 1.0 if pred_val == gt_val else 0.0
    scores["line_items"] = score_line_items(
        pred_json.get("line_items", []),
        gt_json.get("line_items", [])
    )
    return scores

def parse_json_from_output(text):
    """Try to extract the first valid JSON object from model output."""
    # Strip markdown fences if present
    text = re.sub(r'```(?:json)?', '', text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find('{'), text.rfind('}')
        if start != -1 and end != -1:
            try:
                return json.loads(text[start:end+1])
            except json.JSONDecodeError:
                pass
    return {}

print("Scoring helpers ready.")

In [ ]:
def run_inference(model, tokenizer, examples, label="Model"):
    """Run all eval examples through the model and return results."""
    FastLanguageModel.for_inference(model)
    results = []

    for i, ex in enumerate(examples):
        messages = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": f"Extract structured invoice data from the following document:\n\n{ex['document_text']}"}
        ]
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            output_ids = model.generate(
                input_ids=inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode only the newly generated tokens
        generated = tokenizer.decode(output_ids[0][inputs.shape[1]:], skip_special_tokens=True)
        pred_json = parse_json_from_output(generated)
        gt_json   = ex["ground_truth_json"]
        scores    = score_example(pred_json, gt_json)

        results.append({
            "id":         ex.get("id", str(i)),
            "difficulty": ex.get("difficulty", "?"),
            "scores":     scores,
            "parsed_ok":  bool(pred_json)
        })

        if (i+1) % 10 == 0:
            print(f"  [{label}] {i+1}/{len(examples)} done...")

    return results

print("Inference function ready.")

In [ ]:
# ── STEP 1: Evaluate BASE model ──────────────────────────────────────────
print("Loading BASE model...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name      = BASE_MODEL,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,
    load_in_4bit    = True,
)
base_tokenizer = get_chat_template(base_tokenizer, chat_template="chatml")

print("Running inference on BASE model...")
base_results = run_inference(base_model, base_tokenizer, eval_examples, label="BASE")
print("Base model evaluation done!")

In [ ]:
# Free base model memory before loading fine-tuned
import gc
del base_model, base_tokenizer
torch.cuda.empty_cache()
gc.collect()
print("Memory freed.")

In [ ]:
# ── STEP 2: Evaluate FINE-TUNED model ────────────────────────────────────
print("Loading FINE-TUNED model...")
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name      = BASE_MODEL,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,
    load_in_4bit    = True,
)
from peft import PeftModel
ft_model = PeftModel.from_pretrained(ft_model, LORA_PATH)
ft_tokenizer = get_chat_template(ft_tokenizer, chat_template="chatml")

print("Running inference on FINE-TUNED model...")
ft_results = run_inference(ft_model, ft_tokenizer, eval_examples, label="FT")
print("Fine-tuned model evaluation done!")

In [ ]:
# ── STEP 3: Compute & Display Results ────────────────────────────────────
ALL_FIELDS = SCALAR_FIELDS + ["line_items"]

def aggregate(results):
    field_scores = {f: [] for f in ALL_FIELDS}
    parse_ok = 0
    by_difficulty = {"easy": [], "medium": [], "hard": []}
    for r in results:
        if r["parsed_ok"]:
            parse_ok += 1
        for f in ALL_FIELDS:
            field_scores[f].append(r["scores"][f])
        diff = r["difficulty"]
        if diff in by_difficulty:
            avg = sum(r["scores"].values()) / len(r["scores"])
            by_difficulty[diff].append(avg)
    return {
        "parse_rate":    parse_ok / len(results),
        "field_scores":  {f: sum(v)/len(v) for f, v in field_scores.items()},
        "overall":       sum(sum(r["scores"].values()) for r in results) / (len(results) * len(ALL_FIELDS)),
        "by_difficulty": {d: (sum(v)/len(v) if v else 0) for d, v in by_difficulty.items()}
    }

base_agg = aggregate(base_results)
ft_agg   = aggregate(ft_results)

print("\n" + "="*65)
print(f"  {'METRIC':<30} {'BASE':>10} {'FINE-TUNED':>12} {'DELTA':>8}")
print("="*65)

def pct(v): return f"{v*100:.1f}%"
def delta(a, b):
    d = b - a
    sign = "+" if d >= 0 else ""
    return f"{sign}{d*100:.1f}pp"

print(f"  {'Overall Accuracy':<30} {pct(base_agg['overall']):>10} {pct(ft_agg['overall']):>12} {delta(base_agg['overall'], ft_agg['overall']):>8}")
print(f"  {'JSON Parse Rate':<30} {pct(base_agg['parse_rate']):>10} {pct(ft_agg['parse_rate']):>12} {delta(base_agg['parse_rate'], ft_agg['parse_rate']):>8}")
print("-"*65)
print("  BY DIFFICULTY")
for diff in ["easy", "medium", "hard"]:
    b = base_agg['by_difficulty'][diff]
    f = ft_agg['by_difficulty'][diff]
    print(f"  {diff.capitalize():<30} {pct(b):>10} {pct(f):>12} {delta(b,f):>8}")
print("-"*65)
print("  BY FIELD")
for field in ALL_FIELDS:
    b = base_agg['field_scores'][field]
    f = ft_agg['field_scores'][field]
    print(f"  {field:<30} {pct(b):>10} {pct(f):>12} {delta(b,f):>8}")
print("="*65)

# Save results to JSON for later analysis
output = {"base": base_agg, "fine_tuned": ft_agg, "n_eval": len(eval_examples)}
Path("eval_results.json").write_text(json.dumps(output, indent=2))
print("\nFull results saved to eval_results.json — right-click in the file browser to download it.")